# Domain 5: Context Management & Reliability
**Exam**: Claude Certified Architect – Foundations (CCAR-F)
**Weighting**: 15% of exam
**Depth**: deep

## Domain Overview

This domain covers preserving critical information across long conversations, designing
escalation/ambiguity-resolution patterns, propagating errors across multi-agent systems, managing
context during large codebase exploration, designing human review workflows with confidence
calibration, and preserving information provenance in multi-source synthesis.

**Why it matters**: context degrades in long-running agentic sessions in predictable ways (lost in
the middle, verbose tool output crowding out relevant facts); architects must design around this
rather than hope it doesn't happen.

**Key concepts to know**: structured fact extraction, escalation triggers, structured error
propagation, scratchpad/session state patterns, confidence calibration, and claim-source
provenance.

In [ ]:
# Setup: these notebooks demonstrate Claude Agent SDK / Claude API / MCP concepts using
# plain Python simulations of the relevant objects and control flow, so every cell is runnable
# without needing a live Claude Agent SDK installation or API key. Where call_claude() is used,
# it falls back to a simulated response if ANTHROPIC_API_KEY is not set.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception:
        return f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY to call the real API] Prompt: {prompt[:120]}..."

print("call_claude() ready. Concepts below use plain Python to simulate Agent SDK / MCP / Claude Code behavior.")

## Key Concepts

### Manage conversation context to preserve critical information across long interactions

Progressive summarization risks condensing numbers, dates, and customer-stated expectations
into vague summaries. The "lost in the middle" effect means models reliably process the beginning
and end of long inputs but may omit findings from the middle. Extracting key facts into a persistent
"case facts" block (outside summarized history) and trimming verbose tool outputs to only relevant
fields both counteract this.

In [ ]:
def extract_case_facts(conversation_notes: list[str]) -> dict:
    """Persist structured facts outside the summarized/compressible history."""
    facts = {}
    for note in conversation_notes:
        if "order" in note.lower():
            facts["order_id"] = note.split("#")[-1].strip() if "#" in note else facts.get("order_id")
        if "$" in note:
            facts["amount"] = note.split("$")[-1].split()[0]
    return facts

print(extract_case_facts(["Customer mentioned order #4471", "They were told $85 refund was approved"]))

### Design effective escalation and ambiguity resolution patterns

Escalate immediately when a customer explicitly demands a human, or when policy is
ambiguous/silent on the specific request, or when no meaningful progress is possible. Sentiment
and self-reported confidence are unreliable proxies for actual complexity. Multiple matching records
require asking for additional identifiers, not heuristic selection.

In [ ]:
def should_escalate(customer_demands_human: bool, policy_covers_case: bool, can_make_progress: bool) -> bool:
    if customer_demands_human:
        return True
    if not policy_covers_case:
        return True
    if not can_make_progress:
        return True
    return False

print(should_escalate(customer_demands_human=False, policy_covers_case=True, can_make_progress=True))
print(should_escalate(customer_demands_human=False, policy_covers_case=False, can_make_progress=True))  # policy gap

### Implement error propagation strategies across multi-agent systems

Structured error context (failure type, attempted query, partial results, alternatives)
enables intelligent coordinator recovery. Distinguish access failures (needing a retry decision) from
valid empty results (a successful query with no matches). Subagents should attempt local recovery
for transient failures and only propagate errors they cannot resolve.

In [ ]:
def subagent_error_report(failure_type: str, attempted_query: str, partial_results: list) -> dict:
    return {
        "failure_type": failure_type,        # e.g. "timeout" vs "no_matches_found"
        "attempted_query": attempted_query,
        "partial_results": partial_results,
        "is_valid_empty_result": failure_type == "no_matches_found",
    }

print(subagent_error_report("timeout", "AI regulation news 2025", []))
print(subagent_error_report("no_matches_found", "internal query xyz", []))

### Manage context effectively in large codebase exploration

Extended sessions can degrade into inconsistent answers referencing "typical patterns"
instead of specific discovered classes. Scratchpad files persist key findings across context
boundaries. Subagent delegation isolates verbose exploration output. Structured state exports
(manifests) enable crash recovery: each agent exports state, and the coordinator loads the manifest
on resume.

In [ ]:
scratchpad = {
    "discovered_classes": ["OrderProcessor", "RefundHandler"],
    "key_finding": "RefundHandler.validate() has no null check on customer_id",
}

def resume_with_manifest(manifest: dict) -> str:
    return f"Resuming with prior findings: {manifest['key_finding']}, classes: {manifest['discovered_classes']}"

print(resume_with_manifest(scratchpad))

### Design human review workflows and confidence calibration

Aggregate accuracy metrics (e.g., 97% overall) can mask poor performance on specific document
types or fields. Stratified random sampling measures error rates and detects novel error patterns.
Field-level confidence scores, calibrated against labeled validation sets, route review attention
where it's most needed.

In [ ]:
def route_for_review(field_confidence: float, threshold: float = 0.85) -> str:
    return "human review" if field_confidence < threshold else "auto-approve"

extraction_confidences = {"invoice_number": 0.99, "due_date": 0.62, "category": 0.91}
for field, conf in extraction_confidences.items():
    print(field, "->", route_for_review(conf))

### Preserve information provenance and handle uncertainty in multi-source synthesis

Source attribution is lost when findings are compressed without preserving claim-source
mappings. Conflicting statistics from credible sources should be annotated with both values and
their sources, not arbitrarily resolved. Publication/collection dates prevent temporal differences
from being misread as contradictions.

In [ ]:
def synthesize_with_provenance(claims: list[dict]) -> str:
    lines = []
    for c in claims:
        lines.append(f"- {c['claim']} (source: {c['source']}, as of {c['date']})")
    return "\n".join(lines)

conflicting_claims = [
    {"claim": "Market grew 12% in 2024", "source": "Report A", "date": "2024-06"},
    {"claim": "Market grew 8% in 2024", "source": "Report B", "date": "2025-01"},
]
print(synthesize_with_provenance(conflicting_claims))

## Mini Hands-On Project

**Covers**: fact extraction, escalation, error propagation, confidence routing, provenance.

**Scenario**: A multi-agent research assistant is investigating a topic across sources with
conflicting data, some subagent timeouts, and low-confidence extractions.

### Step 1: Extract and persist key facts as the session grows

In [ ]:
notes = ["Client budget is $5000", "Deadline mentioned as end of Q4", "Order #991 was referenced once"]
print(extract_case_facts(notes))

### Step 2: Handle a subagent timeout with structured error propagation

In [ ]:
error_report = subagent_error_report("timeout", "2025 industry benchmark report", partial_results=["found 2 of 5 expected sources"])
print(error_report)
print("Coordinator decision: proceed with partial results and annotate coverage gap." if error_report["partial_results"] else "no partial results, escalate")

### Step 3: Route low-confidence findings to human review and preserve provenance

In [ ]:
findings_confidence = {"claim_1": 0.93, "claim_2": 0.55}
for claim, conf in findings_confidence.items():
    print(claim, "->", route_for_review(conf))

print(synthesize_with_provenance(conflicting_claims))

## Lab Exercises

### Exercise 1: Fact extraction under summarization pressure

**Objective**: Extend extract_case_facts to also capture a stated deadline.

**Produce**: An updated function tested against a 4-note conversation.

In [ ]:
def extract_case_facts_v2(conversation_notes: list[str]) -> dict:
    facts = {}
    for note in conversation_notes:
        if "order" in note.lower():
            facts["order_id"] = note.split("#")[-1].strip() if "#" in note else facts.get("order_id")
        if "$" in note:
            facts["amount"] = note.split("$")[-1].split()[0]
        # TODO: also capture a deadline if the word "deadline" or "by" appears
    return facts

notes = ["Order #883 in question", "Refund of $45 discussed", "Deadline is March 1st for resolution"]
print(extract_case_facts_v2(notes))

# Expected output: facts dict now also includes a deadline-related entry extracted from the third note.

In [ ]:
# STRETCH GOAL: simulate the "lost in the middle" effect by placing the deadline note in the
# middle of a 10-note conversation and checking whether your extraction still catches it.
long_notes = ["filler note"] * 4 + ["Deadline is March 1st for resolution"] + ["filler note"] * 4
print(extract_case_facts_v2(long_notes))

# Expected output: The function still correctly extracts the deadline regardless of its position, since it doesn't rely on summarization.

### Exercise 2: Escalation trigger drill

**Objective**: Apply should_escalate to 4 new scenarios.

**Produce**: 4 escalation decisions with reasoning.

In [ ]:
scenarios = [
    ("Customer explicitly says 'let me talk to a person'", True, True, True),
    ("Policy has no rule for cross-border refunds", False, False, True),
    ("Routine password reset request", False, True, True),
    ("Agent stuck in a loop unable to find the account", False, True, False),
]
# TODO: call should_escalate(customer_demands_human, policy_covers_case, can_make_progress) for each
for desc, demands, covers, progress in scenarios:
    pass

# Expected output: Correctly escalates scenarios 1, 2, and 4; does not escalate scenario 3.

In [ ]:
# STRETCH GOAL: add sentiment as a parameter but do NOT let it affect the decision - demonstrate
# why (per the exam content) sentiment should not be a reliable escalation trigger on its own.
def should_escalate_v2(customer_demands_human, policy_covers_case, can_make_progress, customer_sentiment="neutral"):
    # TODO: implement identically to should_escalate, explicitly ignoring customer_sentiment
    pass
print(should_escalate_v2(False, True, True, customer_sentiment="angry"))

# Expected output: False - an angry sentiment alone should NOT trigger escalation if the other 3 factors don't warrant it.

### Exercise 3: Error propagation for a partial failure

**Objective**: Build a coordinator recovery decision based on a structured error report.

**Produce**: A recovery decision function tested against 2 error reports.

In [ ]:
def coordinator_recovery_decision(error_report: dict) -> str:
    # TODO: if error_report["is_valid_empty_result"], treat as a successful query with no matches
    # (not a failure); otherwise if partial_results exist, proceed with a coverage-gap annotation;
    # otherwise escalate/retry
    pass

print(coordinator_recovery_decision(subagent_error_report("no_matches_found", "query xyz", [])))
print(coordinator_recovery_decision(subagent_error_report("timeout", "query abc", ["partial finding"])))

# Expected output: A valid-empty-result case is treated as success; a timeout with partial results proceeds with a noted gap.

In [ ]:
# STRETCH GOAL: add a third case with failure_type="timeout" and empty partial_results,
# and decide whether to retry or escalate to a human.
print(coordinator_recovery_decision(subagent_error_report("timeout", "query def", [])))

# Expected output: A sensible retry-or-escalate decision when there's a real failure with no usable partial results.

### Exercise 4: Scratchpad-based crash recovery

**Objective**: Design a manifest structure for 3 agents and simulate a coordinator resuming from it.

**Produce**: A manifest dict and a resume function output.

In [ ]:
# TODO: design a manifest dict with entries for 3 agents (web_search, document_analysis, synthesis),
# each including at least a status and key_finding field
manifest = {}  # TODO

def resume_from_manifest(manifest: dict) -> str:
    # TODO: build a summary string listing each agent's status and key_finding
    pass

print(resume_from_manifest(manifest))

# Expected output: A manifest covering all 3 agents and a resume summary that reflects each one's last known state.

In [ ]:
# STRETCH GOAL: simulate one agent crashing mid-task (status="crashed", no key_finding) and
# have resume_from_manifest handle that gracefully instead of erroring.
manifest["web_search"] = {"status": "crashed", "key_finding": None}
print(resume_from_manifest(manifest))

# Expected output: A graceful summary noting the crashed agent needs to be re-invoked, without raising an error.

### Exercise 5: Confidence-based review routing at scale

**Objective**: Route a batch of 10 extractions to auto-approve or human review, then compute the review rate.

**Produce**: A review-rate percentage and the list of items needing review.

In [ ]:
extractions = [
    {"id": i, "confidence": c} for i, c in enumerate([0.99, 0.61, 0.88, 0.72, 0.95, 0.99, 0.55, 0.91, 0.83, 0.97])
]
# TODO: use route_for_review to classify each, then compute what % need human review
needs_review = []  # TODO
review_rate = None  # TODO
print(needs_review)
print(review_rate)

# Expected output: A list of the low-confidence items (below 0.85) and a review rate around 40-50% for this sample.

In [ ]:
# STRETCH GOAL: segment the review rate by whether id is even or odd (simulating 2 different document types)
# to demonstrate how aggregate accuracy can mask per-segment differences.
even_ids = [e for e in extractions if e["id"] % 2 == 0]
odd_ids = [e for e in extractions if e["id"] % 2 != 0]
# TODO: compute review rate separately for even_ids and odd_ids

# Expected output: Two different review rates per segment, illustrating why aggregate-only metrics can be misleading.

## Self-Assessment: Context Management & Reliability

Answer each question, then check the Answers cell below.


**Q1.** What is the 'lost in the middle' effect, and how does structured fact extraction counteract it?

**Q2.** Name two conditions, besides an explicit customer request, that should trigger escalation.

**Q3.** Why is a generic 'search unavailable' error status worse than a structured error report for a coordinator?

**Q4.** What is the purpose of a scratchpad file or state manifest in long-running agent sessions?

**Q5.** Why can an aggregate accuracy metric of 97% still hide a serious quality problem?

## Answers


**A1.** Models may miss information located in the middle of long inputs; extracting key facts into a persistent block outside summarized history preserves them regardless of position.

**A2.** A policy gap/ambiguity for the specific case, and an inability to make meaningful progress (also: multiple ambiguous matching records).

**A3.** It hides context (failure type, what was attempted, partial results) that the coordinator needs to make an intelligent recovery decision.

**A4.** To persist key findings and state across context boundaries or crashes, enabling recovery/resumption without re-exploring from scratch.

**A5.** Because it can mask much lower accuracy on a specific document type or field that is underrepresented in the aggregate, requiring stratified analysis to detect.

---
**Next**: [Capstone Project](./00-capstone.ipynb)
**Back**: [README](./README.md)